# Magyar RAG Chatbot – Kisherceg

Futtató környezet módosítása T4 GPU

In [14]:

!pip install -q transformers accelerate bitsandbytes sentence-transformers chromadb requests


In [15]:

import requests
import chromadb
import torch

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline


## Kisherceg szöveg letöltése

In [16]:

url = "https://raw.githubusercontent.com/minorharpman/ai_prog_pub/main/hirlevel_13/test_little.txt"

text = requests.get(url).text

print(text[:1000])


THE LITTLE PRINCE 



Antoine De Saint-Exupery 




Antoine de Saint-Exupery, who was a French author, journalist and pilot wrote 
The Little Prince in 1943, one year before his death. 

The Little Prince appears to be a simple children’s tale, 
some would say that it is actually a profound and deeply moving tale, 
written in riddles and laced with philosophy and poetic metaphor. 




Once when I was six years old I saw a magnificent picture in a book, called True Stories from 
Nature, about the primeval forest. It was a picture of a boa constrictor in the act of swallowing an 
animal. Here is a copy of the drawing. 

In the book it said: “Boa constrictors swallow their prey whole, without chewing it. After that they 
are not able to move, and they sleep through the six months that they need for digestion.” I 
pondered deeply, then, over the adventures of the jungle. And after some work with a coloured 
pencil I succeeded in making my first drawing. My Drawing


## Szöveg darabolása

In [17]:

def chunk_text(text, chunk_size=500):

    chunks = []

    for i in range(0, len(text), chunk_size):
        chunk = text[i:i+chunk_size]
        chunks.append(chunk)

    return chunks


chunks = chunk_text(text)

print("Chunkok száma:", len(chunks))
print(chunks[0])


Chunkok száma: 188
THE LITTLE PRINCE 



Antoine De Saint-Exupery 




Antoine de Saint-Exupery, who was a French author, journalist and pilot wrote 
The Little Prince in 1943, one year before his death. 

The Little Prince appears to be a simple children’s tale, 
some would say that it is actually a profound and deeply moving tale, 
written in riddles and laced with philosophy and poetic metaphor. 




Once when I was six years old I saw a magnificent picture in a book, called True Stories from


## Magyar embedding modell

In [18]:

embedding_model = SentenceTransformer(
    "BAAI/bge-m3"
)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

## ChromaDB

In [20]:

client = chromadb.Client()

collection = client.create_collection(
    name="littleprince"
)


## Embedding készítés

In [21]:

documents = [
    "passage: " + chunk
    for chunk in chunks
]

embeddings = embedding_model.encode(
    documents,
    show_progress_bar=True
).tolist()

collection.add(
    documents=documents,
    embeddings=embeddings,
    ids=[str(i) for i in range(len(documents))]
)

print("Knowledge base kész.")


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Knowledge base kész.


## Magyar LLM

In [22]:

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)



print("LLM betöltve.")


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

LLM betöltve.


## Retrieval

In [23]:

def retrieve_context(query, n_results=3):

    query_embedding = embedding_model.encode(
        ["query: " + query]
    ).tolist()[0]

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )

    docs = results["documents"][0]

    clean_docs = [
        d.replace("passage: ", "")
        for d in docs
    ]

    return "\n".join(clean_docs)


## Chatbot

In [26]:
def ask_bot(question):

    context = retrieve_context(question)

    prompt_off = f"""
Te egy magyar nyelvű AI asszisztens vagy.

CSAK a megadott szöveg alapján válaszolj.

Ha nincs válasz a szövegben,
akkor mondd azt:

"Nincs információm erről a szöveg alapján."

Szöveg:
{context}

Kérdés:
{question}

Válasz:
"""
    # finomítani, hogy csak a szövegből válaszoljon.
    prompt_off2 = f"""
    You are a friendly and natural conversational AI assistant.

    Use the provided context as background knowledge,
    but do not quote it word-for-word unless necessary.

    Answer naturally and conversationally,
    like ChatGPT.

    If the context is relevant, use it.
    If not, answer normally using your general knowledge.

    Context:
    {context}

    User:
    {question}

    Assistant:
    """

    prompt = f"""
    You are a friendly and natural conversational AI assistant.

    Answer ONLY using the provided context.

    If the answer cannot be found in the context,
    then say:

    "This question is not related to The Little Prince context."

    Do not use external knowledge.
    Do not make up information.
    Do not answer from memory.

    Answer naturally and conversationally.

    Context:
    {context}

    User:
    {question}

    Assistant:
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    answer = response[len(prompt):]

    return answer.strip()


## Chat loop

In [27]:

while True:

    q = input("You: ")

    if q.lower() == "exit":
        break

    response = ask_bot(q)

    print("\nAI:\n")
    print(response)
    print("\n")


Te: where is the lamb?

AI:

The text does not mention anything about a lamb. Therefore, based on the given context, the answer is:

    There is no mention of a lamb in the provided context. 

The context discusses the story's narrator reflecting on their feelings towards the little prince after discovering that the shepherd had forgotten to put a leather strap on his sheep, which prevented the sheep from being able to fasten its ears to the sheepdog's ear loops. The narrator wonders what may have happened to the little prince on his planet, considering the possibility that the sheep could have eaten a rose, but ultimately concludes that this is just a mystery without providing further details. There is no reference to a lamb in the passage.


Te: where is the fox?

AI:

Under the little prince's feet.


Te: what did the foy say?

AI:

The fox said, "I am all alone... all alone..." This repeated phrase indicates his loneliness and isolation. It's worth noting that the fox refers to hi